In [48]:
import trimesh
from pathlib import Path


# define the paths
model_id = "107bce22d72f322eedf1bb0b62653056"
root_dir = Path("/home/borth/2d-gaussian-splatting/")
shapenet_dir = root_dir / "data/ShapeNetCore/04256520"
shapenet_path = shapenet_dir / model_id / "models/model_normalized.obj"

paths = [p / "models/model_normalized.obj" for p in shapenet_dir.iterdir()]
is_watertight = 0
for path in paths[:10]:
    mesh = trimesh.load(shapenet_path)
    mesh.process()
    mesh.fix_normals()
    if mesh.is_watertight:
        is_watertight += 1
is_watertight

0

In [52]:
mesh.remove_degenerate_faces()

/tmp/ipykernel_468580/3986485565.py:1: DeprecationWarning: `remove_degenerate_faces` is deprecated and will be removed in March 2024 replace with `self.update_faces(self.nondegenerate_faces(height=height))`
  mesh.remove_degenerate_faces()


In [53]:
# Fill holes
mesh.fill_holes()

# Remove degenerate faces
mesh.remove_degenerate_faces()

# Merge close vertices
mesh = mesh.simplify_quadratic_decimation(mesh.faces.shape[0] - 10)

# Check again
print("Is the mesh watertight now?", mesh.is_watertight)


Is the mesh watertight now? False


/tmp/ipykernel_468580/2188465404.py:5: DeprecationWarning: `remove_degenerate_faces` is deprecated and will be removed in March 2024 replace with `self.update_faces(self.nondegenerate_faces(height=height))`
  mesh.remove_degenerate_faces()
/tmp/ipykernel_468580/2188465404.py:8: DeprecationWarning: `simplify_quadratic_decimation` is deprecated as it was a typo and will be removed in March 2024: replace with `simplify_quadric_decimation`
  mesh = mesh.simplify_quadratic_decimation(mesh.faces.shape[0] - 10)


In [31]:
import torch
import time
points = torch.rand(1_000_000, 3)

start = torch.cuda.Event(enable_timing=True)
end= torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(10):
    idx = torch.randperm(len(points))[:100_000]
    s = points[idx]
end.record()
torch.cuda.synchronize()
print(start.elapsed_time(end)/ 10)


21.01299133300781


In [ ]:
print("Number of boundary edges:", len(mesh.edges_boundary))



AttributeError: 'Trimesh' object has no attribute 'edges_boundary'

In [28]:
import trimesh

# Create a watertight dummy mesh (an icosphere)
mesh = trimesh.creation.icosphere(subdivisions=3)

# Check if it's watertight
print("Is the dummy mesh watertight?", mesh.is_watertight)

Is the dummy mesh watertight? True


In [39]:
import open3d as o3d
import numpy as np

# Load a mesh (replace with your file)
mesh = o3d.io.read_triangle_mesh(str(path))

# Compute adjacency and edge properties
mesh.compute_adjacency_list()
edges = np.asarray(mesh.get_non_manifold_edges())

# If there are no non-manifold edges, the mesh is likely watertight
is_watertight = len(edges) == 0
print("Is the mesh watertight?", is_watertight)


Is the mesh watertight? True


AttributeError: 'open3d.cuda.pybind.geometry.TriangleMesh' object has no attribute 'fill_holes'